In [ ]:
# Source - https://stackoverflow.com/a/53094059
# Posted by H-San, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-19, License - CC BY-SA 4.0

from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
pics_dir = "/content/drive/My Drive/DL_Data/"
csv_dir = "/content/drive/My Drive/DL_Data/train.csv"

In [ ]:
import pandas as pd
import os
from PIL import Image
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import random
import cv2
import numpy as np

In [ ]:
# setting the seed
torch.manual_seed(42)

In [ ]:
train_dir = "/content/drive/My Drive/DL_Data/split_blanche/train_images/"
test_dir = "/content/drive/My Drive/DL_Data/split_blanche/test_images/"

In [ ]:
train_df = pd.read_csv(os.path.join("/content/drive/My Drive/DL_Data/split_blanche/", "train_blanche.csv"))
test_df = pd.read_csv(os.path.join("/content/drive/My Drive/DL_Data/split_blanche/", "test_blanche.csv"))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/DL_Data/split_blanche/train_blanche.csv'

Loading the data

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, self.df.columns.get_loc('image_path')]
        img_path = os.path.join(self.img_dir, img_name) # img name should be sth like train/id
        image = Image.open(img_path).convert('RGB')
        target = self.df.iloc[idx, self.df.columns.get_loc('target')]

        if self.transform:
            image = self.transform(image)

        return image, img_path

    def get_csv(self):
      return self.df

In [ ]:
custom_dataset = CustomImageDataset(csv_file=csv_dir, img_dir=pics_dir, transform=None)

In [ ]:
df = custom_dataset.get_csv()
df.head()

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750


making a df of unique image paths (so that we get only the image names)

In [ ]:
import numpy as np
import pandas as pd

unique_image_paths = df['image_path'].unique()
print(f"Total images:    {len(unique_image_paths)}")
print(f"Total rows:      {len(df)}")
print(f"Rows per image:  {len(df) / len(unique_image_paths):.1f}")


Total images:    357
Total rows:      1785
Rows per image:  5.0


# Applying the train-test split on the CSV

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset # Import Subset

# they create numpy arrays
train_img_paths, test_img_paths = train_test_split(
    unique_image_paths,
    test_size=0.2,
    shuffle=True,
    random_state=42,
)


In [ ]:
print(len(train_img_paths))
print(len(test_img_paths))
print("they should sum up to the total number of images; ", len(train_img_paths) + len(test_img_paths))
type(train_img_paths)

285
72
they should sum up to the total number of images;  357


numpy.ndarray

In [ ]:
# Convert numpy arrays to sets for fast lookup
train_paths = set(train_img_paths)
test_paths   = set(test_img_paths)

# Filter original df
train_df = df[df['image_path'].isin(train_paths)].reset_index(drop=True)
test_df = df[df['image_path'].isin(test_paths)].reset_index(drop=True)

train_df.to_csv('train_blanche.csv', index=False)
test_df.to_csv('test_blanche.csv',   index=False)

# Sanity checks...!!!
print(f"Total rows:      {len(df)}")
print(f"Train rows:      {len(train_df)}")
print(f"Val rows:        {len(test_df)}")
print(f"Row sum matches: {len(train_df) + len(test_df) == len(df)}")

# Make sure no image appears in both splits
assert len(train_paths & test_paths) == 0, "Leakage detected!"

Total rows:      1785
Train rows:      1425
Val rows:        360
Row sum matches: True


In [ ]:
test_df.head()

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1028611175__Dry_Clover_g,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0,Dry_Clover_g,0.0000
1,ID1028611175__Dry_Dead_g,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0,Dry_Dead_g,30.9703
2,ID1028611175__Dry_Green_g,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0,Dry_Green_g,24.2376
3,ID1028611175__Dry_Total_g,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0,Dry_Total_g,55.2079
4,ID1028611175__GDM_g,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0,GDM_g,24.2376


sanity check;

In [ ]:
train_counts = train_df.groupby('image_path').size()
test_counts   = test_df.groupby('image_path').size()

# Are all exactly 4?
print(f"Train — all images have 4 rows: {(train_counts == 5).all()}")
print(f"Val   — all images have 4 rows: {(test_counts == 5).all()}")

bad_train = train_counts[train_counts != 4]
bad_val   = test_counts[test_counts != 4]

if len(bad_train) > 0:
    print(f"Train bad counts:\n{bad_train}")
if len(bad_val) > 0:
    print(f"Val bad counts:\n{bad_val}")

Train — all images have 4 rows: True
Val   — all images have 4 rows: True
Train bad counts:
image_path
train/ID1011485656.jpg    5
train/ID1012260530.jpg    5
train/ID1025234388.jpg    5
train/ID1035947949.jpg    5
train/ID1049634115.jpg    5
                         ..
train/ID969218269.jpg     5
train/ID972274220.jpg     5
train/ID975115267.jpg     5
train/ID980538882.jpg     5
train/ID983582017.jpg     5
Length: 285, dtype: int64
Val bad counts:
image_path
train/ID1028611175.jpg    5
train/ID1036339023.jpg    5
train/ID105271783.jpg     5
train/ID1084819986.jpg    5
train/ID1119739385.jpg    5
                         ..
train/ID94564238.jpg      5
train/ID950496197.jpg     5
train/ID95050718.jpg      5
train/ID978026131.jpg     5
train/ID980878870.jpg     5
Length: 72, dtype: int64


# Preprocessing methods

*   Removing the date stamp
*   Normalizing RGB values across the dataset -> This keeps the "numbers" small and helps the model converge faster.
* we also explored what other participants did in the competiton;


https://www.kaggle.com/competitions/csiro-biomass/writeups/physics-constrained-regression-integrating-biolog

In [ ]:
# disclaimer: this function was taken from the competition's discussion forum
def clean_image(img):
    """
    Image preprocessing: Remove bottom artifacts and date stamp.
    """
    # Detect orange date stamp in HSV space
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    # Define orange range
    lower = np.array([5, 150, 150])
    upper = np.array([25, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)

    # Dilate mask to cover text edges
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=2)

    # Inpaint if timestamp detected
    if np.sum(mask) > 0:
        img = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)

    return img

In [ ]:
from torchvision import transforms

def color_jitter(image, threshold):
    """
    Apply random color jittering to a PIL Image.

    Args:
        image: PIL Image
    Returns:
        Augmented PIL Image
    """
    if random.random() < threshold:
        print("I am jittering")
        # Define the color jitter transform
        transform = transforms.ColorJitter(
            brightness=0.2,  # ± 20% brightness
            contrast=0.2,    # ± 20% contrast
            saturation=0.2,  # ± 20% saturation
            hue=0.1,          # ± 10% hue shift
        )
        image = transform(image)

    return image

In [ ]:
def apply_random_transforms(image, threshold):
    """
    Apply random rotation and flipping to a PIL Image.

    Args:
        image: PIL Image
        threshold: Probability threshold for applying each transformation.
    Returns:
        Augmented PIL Image
    """
    if random.random() < threshold:
        print("I am rotating")
        angles = [90, 180, 270]
        angle = random.choice(angles)
        image = image.rotate(angle, expand=True) # expand=True to avoid cropping

    if random.random() < threshold:
        print("I am flipping left to right")
        image = image.transpose(Image.FLIP_LEFT_RIGHT)

    if random.random() < threshold:
        print("I am flipping up to down")
        image = image.transpose(Image.FLIP_TOP_BOTTOM)

    return image

The following code is really important! We used it to normalize the picturessss

In [ ]:
import cv2
import os
import numpy as np
from tqdm import tqdm

def calculate_dataset_stats(image_folder, img_size=224):
    pixel_sum = np.zeros(3)
    pixel_sq_sum = np.zeros(3)
    pixel_count = 0

    image_files = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]

    for file in tqdm(image_files, desc="Calculating Stats"):
        img = cv2.imread(os.path.join(image_folder, file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0  # Switch to RGB and 0-1 range
        # img = cv2.resize(img, (img_size, img_size))

        pixel_sum += np.sum(img, axis=(0, 1))
        pixel_sq_sum += np.sum(np.square(img), axis=(0, 1))
        pixel_count += img.shape[0] * img.shape[1]

    mean = pixel_sum / pixel_count
    std = np.sqrt((pixel_sq_sum / pixel_count) - np.square(mean))

    return mean, std

train_mean, train_std = calculate_dataset_stats(train_dir)
print(f"Mean: {train_mean}, Std: {train_std}")

Calculating Stats: 100%|██████████| 570/570 [01:16<00:00,  7.44it/s]

Mean: [0.43918201 0.49828457 0.30579902], Std: [0.24945421 0.24771695 0.23592069]


In [ ]:
def normalize_image(image: Image, mean, std):
    img_array = np.array(image, dtype=np.float32)

    img_normalized = img_array / 255.0 # to get it between 0 and 1 basically

    img_final = (img_normalized - mean) / std

    return img_final

# Train-test split on images and preprocessing

## Train set

In [ ]:
train_dataset = CustomImageDataset(csv_file='train_blanche.csv', img_dir=pics_dir, transform=None)

In [ ]:
output_dir = "/content/drive/My Drive/DL_Data/split_blanche/train_images"
os.makedirs(output_dir, exist_ok=True)

# rsize transform
resize_transform = transforms.Resize(size=(256, 256), interpolation=transforms.InterpolationMode.BICUBIC)

# as we saw in the EDA, all the images are the same size (2000x1000). We decided to split them in 2 to get (1000x1000)
for img_path in train_df['image_path'].unique():

    pil_image     = Image.open(os.path.join(pics_dir, img_path)).convert('RGB')
    image = Image.fromarray(clean_image(np.array(pil_image))) # we clean the image

    w, h  = image.size
    left  = image.crop((0, 0, w//2, h))
    right = image.crop((w//2, 0, w, h))


    image = color_jitter(image, 0.5) # we apply the color jittering
    image = apply_random_transforms(image, 0.3) # we apply the transforms


    # Apply resize transform and then normalize https://datascience.stackexchange.com/questions/44330/normalization-before-or-after-resizing
    left = resize_transform(left)
    right = resize_transform(right)

    base = os.path.splitext(os.path.basename(img_path))[0]

    left.save(os.path.join(output_dir,  f"{base}_L.jpg"), quality=95)
    right.save(os.path.join(output_dir, f"{base}_R.jpg"), quality=95)

print(f"Done — saved to {output_dir}")

I am jittering
I am flipping left to right
I am jittering
I am flipping left to right
I am flipping up to down
I am rotating
I am flipping up to down
I am rotating
I am rotating
I am jittering
I am flipping up to down
I am rotating
I am flipping up to down
I am jittering
I am rotating
I am jittering
I am rotating
I am rotating
I am flipping left to right
I am rotating
I am flipping left to right
I am jittering
I am rotating
I am rotating
I am flipping left to right
I am flipping left to right
I am flipping up to down
I am jittering
I am flipping left to right
I am rotating
I am jittering
I am rotating
I am flipping up to down
I am jittering
I am flipping up to down
I am jittering
I am rotating
I am jittering
I am flipping left to right
I am jittering
I am flipping left to right
I am flipping up to down
I am jittering
I am rotating
I am flipping up to down
I am jittering
I am flipping up to down
I am jittering
I am rotating
I am flipping left to right
I am flipping up to down
I am jitte

In [ ]:
from pathlib import Path

output_dir = "drive/MyDrive/DL_Data/split_blanche/train_images"

def check_all_good(output_dir, df):
    # initializing the counterss
    total       = 0
    wrong_size  = 0
    corrupted   = 0

    for img_file in Path(output_dir).glob("*.jpg"):
        total += 1
        try:
            img = Image.open(img_file)
            w, h = img.size

            if (w, h) != (256, 256):
                wrong_size += 1
                print(f"Wrong size {w}x{h}: {img_file.name}")

        except Exception as e:
            corrupted += 1
            print(f"Corrupted: {img_file.name} — {e}")

    print(f"\nTotal images:  {total}")
    print(f"Expected:      {len(df['image_path'].unique()) * 2}")  # *2 because L and R
    print(f"Wrong size:    {wrong_size}")
    print(f"Corrupted:     {corrupted}")
    print(f"All good:      {wrong_size == 0 and corrupted == 0}")


check_all_good(output_dir, train_df)


Total images:  570
Expected:      570
Wrong size:    0
Corrupted:     0
All good:      True


## test set

In [ ]:
output_dir = "drive/MyDrive/DL_Data/split_blanche/test_images"
os.makedirs(output_dir, exist_ok=True)

for img_path in test_df['image_path'].unique():

    pil_image     = Image.open(os.path.join(pics_dir, img_path)).convert('RGB')
    cleaned_image = Image.fromarray(clean_image(np.array(pil_image)))

    w, h  = cleaned_image.size
    left  = cleaned_image.crop((0, 0, w//2, h))
    right = cleaned_image.crop((w//2, 0, w, h))

    base = os.path.splitext(os.path.basename(img_path))[0]
    left.save(os.path.join(output_dir,  f"{base}_L.jpg"), quality=95)
    right.save(os.path.join(output_dir, f"{base}_R.jpg"), quality=95)

print(f"Done — saved to {output_dir}")

Done — saved to drive/MyDrive/DL_Data/split_blanche/test_images


In [ ]:
output_dir = "drive/MyDrive/DL_Data/split_blanche/test_images"
check_all_good(output_dir, test_df)


Total images:  144
Expected:      144
Wrong size:    0
Corrupted:     0
All good:      True


# more preprocessing - on the csv files

---



In [ ]:
def expand_csv_for_splits(csv_path, output_path):
    """
    Duplicates each row for the L and R splits of each image.

    Args:
        csv_path:    path to original csv
        output_path: path to save expanded csv
    """
    df = pd.read_csv(csv_path)

    rows = []
    for _, row in df.iterrows():
        base = os.path.splitext(row['image_path'])[0]  # remove .jpg

        for side in ['L', 'R']:
            new_row = row.copy()
            new_row['image_path'] = f"{base}_{side}.jpg"
            rows.append(new_row)

    expanded_df = pd.DataFrame(rows).reset_index(drop=True)
    expanded_df.to_csv(output_path, index=False)

    print(f"Original rows:  {len(df)}")
    print(f"Expanded rows:  {len(expanded_df)}")  # should be exactly 2x
    print(f"Saved to:       {output_path}")
    return expanded_df

# Run on both
train_df_expanded = expand_csv_for_splits('/content/drive/My Drive/DL_Data/split_blanche/train_blanche.csv', 'train_split.csv')
test_df_expanded  = expand_csv_for_splits('/content/drive/My Drive/DL_Data/split_blanche/test_blanche.csv',  'test_split.csv')

Original rows:  1425
Expanded rows:  2850
Saved to:       train_split.csv
Original rows:  360
Expanded rows:  720
Saved to:       test_split.csv


In [ ]:
train_split_df = pd.read_csv('train_split.csv')
test_split_df  = pd.read_csv('test_split.csv')

In [ ]:
train_split_df.head()

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656_L.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Clover_g,train/ID1011485656_R.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
2,ID1011485656__Dry_Dead_g,train/ID1011485656_L.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
3,ID1011485656__Dry_Dead_g,train/ID1011485656_R.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
4,ID1011485656__Dry_Green_g,train/ID1011485656_L.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751


# Model preparation!!!
* We took the mean and std of the train images so that we can also apply it to the test set later on. This function cell will be used in many of our model notebooks. This is because we did not want to save the images normalized and instead, we normalize them before running them.

In [ ]:
train_mean = [0.43918201, 0.49828457, 0.30579902]
train_std = [0.24945421, 0.24771695, 0.23592069]

def normalize_image(image: Image, mean, std):
    img_array = np.array(image, dtype=np.float32)
    img_normalized = img_array / 255.0

    img_final = (img_normalized - mean) / std

    return img_final